# KS0223 Presentation Notebook

This notebook is for live demos of the Unity simulator:
- API sanity check (`health`, `contract`)
- camera frame preview
- short scripted drive with telemetry plots
- optional manual control cell

Before running:
1. Open Unity project and press `Play`.
2. Ensure HTTP API host is enabled on `http://127.0.0.1:8000`.


In [ ]:
# Optional: uncomment if your environment is missing packages
# %pip install -q requests matplotlib numpy ipywidgets

from __future__ import annotations

import base64
import io
import math
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

ROOT = Path.cwd()
PYTHON_DIR = ROOT / "python"
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))

from sim_client.http_client import SimClient
from sim_client.ks0223 import Ks0223Command, parse_telemetry

BASE_URL = "http://127.0.0.1:8000"
client = SimClient(BASE_URL, timeout_s=20.0)

print(f"Using API: {BASE_URL}")


In [ ]:
health = client.health()
contract = client.get_contract()

print("health:", health)
print("simulator:", contract.get("simulatorName"), "contract:", contract.get("contractVersion"))
print("vehicles:", [v.get("deviceId") for v in contract.get("availableVehicles", [])])
print("tracks:", [t.get("trackId") for t in contract.get("availableTracks", [])])


In [ ]:
def _frame_bytes(step_result: dict[str, Any]) -> bytes:
    frame = step_result.get("frame") or {}
    b64 = frame.get("dataBase64") or frame.get("data") or ""
    return base64.b64decode(b64) if b64 else b""


def frame_to_array(step_result: dict[str, Any]) -> np.ndarray | None:
    payload = _frame_bytes(step_result)
    if not payload:
        return None
    return mpimg.imread(io.BytesIO(payload), format="jpg")


def kvf(telemetry: dict[str, str], key: str, default: float = float("nan")) -> float:
    value = telemetry.get(key)
    if value is None:
        return default
    try:
        return float(value)
    except ValueError:
        return default


def step_pwm(left: float, right: float, brake: float = 0.0, delta_time: float = 0.1) -> dict[str, Any]:
    cmd = Ks0223Command(left_pwm_norm=left, right_pwm_norm=right, brake=brake).to_step_command()
    return client.step({"action": cmd, "deltaTime": delta_time})


In [ ]:
_ = client.reset({})
sample = step_pwm(0.0, 0.0, brake=1.0, delta_time=0.1)
telemetry = parse_telemetry(sample)
img = frame_to_array(sample)

print("speed m/s:", round((sample.get("state") or {}).get("speed", 0.0), 3))
print("telemetry keys:", sorted(list(telemetry.keys()))[:8], "...")

if img is None:
    print("No frame in response. Check camera sensor in Unity scene.")
else:
    plt.figure(figsize=(8, 4.5))
    plt.imshow(img)
    plt.title("Front camera preview")
    plt.axis("off")
    plt.show()


In [ ]:
# Scripted demo profile: straight -> left turn -> straight -> right correction
profile = []
profile += [(0.35, 0.35, 0.0)] * 20
profile += [(0.20, 0.42, 0.0)] * 18
profile += [(0.34, 0.34, 0.0)] * 15
profile += [(0.42, 0.22, 0.0)] * 12
profile += [(0.0, 0.0, 0.6)] * 6

_ = client.reset({})
records = []
frames = []

for i, (left, right, brake) in enumerate(profile):
    step = step_pwm(left, right, brake=brake, delta_time=0.1)
    telemetry = parse_telemetry(step)
    state = step.get("state") or {}

    records.append(
        {
            "i": i,
            "left_pwm": left,
            "right_pwm": right,
            "speed": float(state.get("speed", 0.0)),
            "ultrasonic_m": kvf(telemetry, "sensor.ultrasonic.front.m"),
            "voltage_v": kvf(telemetry, "power.battery.voltage_v"),
            "current_a": kvf(telemetry, "power.battery.current_a"),
            "line_s3": kvf(telemetry, "sensor.line_tracker.s3_norm"),
        }
    )

    # Keep sparse snapshots for presentation slides.
    if i in (0, 20, 38, len(profile) - 1):
        frame = frame_to_array(step)
        if frame is not None:
            frames.append((i, frame))

print(f"Collected {len(records)} steps, snapshots: {len(frames)}")


In [ ]:
arr_i = np.array([r["i"] for r in records], dtype=float)
arr_speed = np.array([r["speed"] for r in records], dtype=float)
arr_ultra = np.array([r["ultrasonic_m"] for r in records], dtype=float)
arr_voltage = np.array([r["voltage_v"] for r in records], dtype=float)
arr_line = np.array([r["line_s3"] for r in records], dtype=float)

fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
axes[0, 0].plot(arr_i, arr_speed)
axes[0, 0].set_title("Speed (m/s)")
axes[0, 0].set_xlabel("step")

axes[0, 1].plot(arr_i, arr_ultra)
axes[0, 1].set_title("Ultrasonic front (m)")
axes[0, 1].set_xlabel("step")

axes[1, 0].plot(arr_i, arr_voltage)
axes[1, 0].set_title("Battery voltage (V)")
axes[1, 0].set_xlabel("step")

axes[1, 1].plot(arr_i, arr_line)
axes[1, 1].set_title("Line tracker S3 (norm)")
axes[1, 1].set_xlabel("step")

plt.show()

if frames:
    fig, axs = plt.subplots(1, len(frames), figsize=(4 * len(frames), 3), constrained_layout=True)
    if len(frames) == 1:
        axs = [axs]
    for ax, (idx, img) in zip(axs, frames):
        ax.imshow(img)
        ax.set_title(f"step {idx}")
        ax.axis("off")
    plt.show()


## Optional: manual control widget

This cell creates sliders for left/right PWM and brake. Use it live during a demo.
If `ipywidgets` is missing, run the install command from the first cell.


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
except Exception as exc:
    print("ipywidgets not available:", exc)
else:
    left = widgets.FloatSlider(description="left", min=-1.0, max=1.0, step=0.01, value=0.0)
    right = widgets.FloatSlider(description="right", min=-1.0, max=1.0, step=0.01, value=0.0)
    brake = widgets.FloatSlider(description="brake", min=0.0, max=1.0, step=0.01, value=0.0)
    delta = widgets.FloatSlider(description="dt", min=0.05, max=0.5, step=0.05, value=0.1)
    out = widgets.Output()

    def run_once(_):
        with out:
            out.clear_output(wait=True)
            step = step_pwm(left.value, right.value, brake.value, delta.value)
            telemetry = parse_telemetry(step)
            speed = (step.get("state") or {}).get("speed")
            print({
                "speed": speed,
                "voltage_v": telemetry.get("power.battery.voltage_v"),
                "current_a": telemetry.get("power.battery.current_a"),
                "ultrasonic_m": telemetry.get("sensor.ultrasonic.front.m"),
            })
            img = frame_to_array(step)
            if img is not None:
                plt.figure(figsize=(6, 3.2))
                plt.imshow(img)
                plt.axis("off")
                plt.show()

    btn = widgets.Button(description="Send Step", button_style="primary")
    btn.on_click(run_once)
    display(widgets.VBox([widgets.HBox([left, right, brake, delta]), btn, out]))


## ROS2 quick note

ROS2 is optional. Core simulator logic and training can run without ROS2.
Use ROS2 when you need ecosystem interoperability (RViz/rqt/topics/services, robotics stacks, and real robot bridge).

Current bridge in this repo:
- `python/bridges/ros2_bridge.py`
- publish: `state_json`, `telemetry_json`, `camera/front/image_raw/compressed`
- subscribe: `cmd_drive`, `cmd_drive_json`
